# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ruchi74/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis:** One row represents one content item for one client on one reporting date (`report_date × client_hash_id × content_hash_id`).

**Time window:** March 1–31, 2026. This month was selected as a mid-panel slice for verification. The partition contains 9,841,378 rows, covering the full month.

**Grain verification:** No duplicate combinations were found for the selected grain in the March partition.

**Availability and missingness:** All rows have GSC availability, while 6,822,637 rows have GA4 availability. No missing report dates, client IDs, or content IDs were found in this partition. GA4 metrics will only be interpreted when `client_has_ga4 IS TRUE`.

In [4]:

%pip -q install duckdb huggingface_hub


In [5]:

from google.colab import userdata
import duckdb

token = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")
con.execute("INSTALL huggingface")
con.execute("LOAD huggingface")

con.execute(
    "CREATE OR REPLACE SECRET hf "
    "(TYPE huggingface, TOKEN ?)",
    [token]
)

print("DuckDB is ready.")


HTTPException: HTTP Error: Failed to download extension "huggingface" at URL "http://extensions.duckdb.org/v1.3.2/linux_amd64/huggingface.duckdb_extension.gz" (HTTP 404)

Candidate extensions: "ui", "inet", "ducklake", "autocomplete", "httpfs"
For more info, visit https://duckdb.org/docs/stable/extensions/troubleshooting/?version=v1.3.2&platform=linux_amd64&extension=huggingface

In [6]:

%pip -q install --upgrade duckdb huggingface_hub


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.41.0 requires huggingface-hub<2.0,>=1.32.0, but you have huggingface-hub 2.3.0 which is incompatible.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.3.0 which is incompatible.


In [7]:

%pip install -q "huggingface-hub>=1.32.0,<2.0"


In [1]:

from google.colab import userdata
from huggingface_hub import HfApi
import duckdb

token = userdata.get("HF_TOKEN")

api = HfApi(token=token)
print("Hugging Face:", api.whoami()["name"])

con = duckdb.connect()
con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

print("DuckDB version:", duckdb.__version__)
print("Connection setup complete.")


Hugging Face: Ruchita74
DuckDB version: 2.0.0
Connection setup complete.


In [3]:

import duckdb

con = duckdb.connect()
con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

url = (
    "https://huggingface.co/datasets/"
    "FlyRank/internship-warehouse/resolve/main/"
    "README.md"
)

try:
    result = con.execute(
        "SELECT * FROM read_csv_auto(?) LIMIT 5",
        [url]
    ).fetchall()
    print(result)
except Exception as e:
    print("Test result:", type(e).__name__)
    print(str(e)[:1000])


[('license: other',), ('language:',), ('- en',), ('tags:',), ('- seo',)]


In [4]:

from huggingface_hub import hf_hub_download
import os

token = userdata.get("HF_TOKEN")

try:
    path = hf_hub_download(
        repo_id="FlyRank/internship-warehouse",
        filename="README.md",
        repo_type="dataset",
        token=token
    )
    print("Authenticated dataset access works!")
    print("Metadata file downloaded:", os.path.basename(path))
except Exception as e:
    print("Access test:", type(e).__name__)
    print(str(e)[:1000])


README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

Authenticated dataset access works!
Metadata file downloaded: README.md


In [5]:

from huggingface_hub import HfApi

api = HfApi(token=token)

files = api.list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset"
)

for f in files:
    if "fact_content_daily_performance" in f:
        print(f)


fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/month=2026-04/data_0.parquet
fact_content_daily_performance/month=202

In [6]:

from huggingface_hub import hf_hub_url
import duckdb

token = userdata.get("HF_TOKEN")

march_url = hf_hub_url(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset"
)

con = duckdb.connect()
con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN '{token}'
)
""")

query = f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet('{march_url}')
"""

print(con.execute(query).df())


HTTPException: HTTP Error: HTTP GET error on 'https://huggingface.co/datasets/FlyRank/internship-warehouse/resolve/main/fact_content_daily_performance/month%3D2026-03/data_0.parquet' (HTTP 401 Unauthorized)

LINE 5: FROM read_parquet('https://huggingface.co/datasets/FlyRank/intern...
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

In [7]:

from google.colab import userdata
import duckdb

token = userdata.get("HF_TOKEN")
con = duckdb.connect()

con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

con.execute("DROP SECRET IF EXISTS hf")

con.execute("""
CREATE SECRET hf (
    TYPE http,
    EXTRA_HTTP_HEADERS MAP {
        'Authorization': 'Bearer """ + token + """'
    }
)
""")

print("Authenticated HTTP access configured.")


Authenticated HTTP access configured.


In [8]:

march_url = (
    "https://huggingface.co/datasets/FlyRank/"
    "internship-warehouse/resolve/main/"
    "fact_content_daily_performance/month%3D2026-03/data_0.parquet"
)

query = f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet('{march_url}')
"""

try:
    print(con.execute(query).df())
except Exception as e:
    print(type(e).__name__)
    print(str(e)[:1000])


   row_count first_date  last_date
0    9841378 2026-03-01 2026-03-31


In [9]:

grain_query = f"""
SELECT
    report_date,
    client_id,
    content_id,
    COUNT(*) AS row_count
FROM read_parquet('{march_url}')
GROUP BY report_date, client_id, content_id
HAVING COUNT(*) > 1
LIMIT 5
"""

duplicates = con.execute(grain_query).df()

if duplicates.empty:
    print("PASS: No duplicate rows found for the tested grain.")
else:
    print("Duplicate combinations found:")
    print(duplicates)


BinderException: Binder Error: Referenced column "client_id" not found in FROM clause!
Candidate bindings: "client_hash_id", "client_has_ga4", "client_has_gsc", "content_hash_id", "sessions_paid"

LINE 7: GROUP BY report_date, client_id, content_id
                               ^^^^^^^^^

In [10]:

grain_query = f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM read_parquet('{march_url}')
GROUP BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
"""

duplicates = con.execute(grain_query).df()

if duplicates.empty:
    print("PASS: No duplicate rows found for the tested grain.")
else:
    print("Duplicate combinations found:")
    print(duplicates)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

PASS: No duplicate rows found for the tested grain.


In [11]:

availability_query = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE client_has_gsc IS TRUE
    ) AS gsc_available_rows,
    COUNT(*) FILTER (
        WHERE client_has_ga4 IS TRUE
    ) AS ga4_available_rows,
    COUNT(*) FILTER (
        WHERE report_date IS NULL
    ) AS missing_dates,
    COUNT(*) FILTER (
        WHERE client_hash_id IS NULL
    ) AS missing_client_ids,
    COUNT(*) FILTER (
        WHERE content_hash_id IS NULL
    ) AS missing_content_ids
FROM read_parquet('{march_url}')
"""

print(con.execute(availability_query).df())


   total_rows  gsc_available_rows  ga4_available_rows  missing_dates  \
0     9841378             9841378             6822637              0   

   missing_client_ids  missing_content_ids  
0                   0                    0  


In [12]:

schema_query = f"""
DESCRIBE
SELECT *
FROM read_parquet('{march_url}')
"""

print(con.execute(schema_query).df().to_string(index=False))


             column_name column_type null  key default extra
             report_date        DATE  YES None    None  None
          client_hash_id     VARCHAR  YES None    None  None
         content_hash_id     VARCHAR  YES None    None  None
          client_has_gsc     BOOLEAN  YES None    None  None
          client_has_ga4     BOOLEAN  YES None    None  None
      gsc_data_available     BOOLEAN  YES None    None  None
      ga4_data_available     BOOLEAN  YES None    None  None
         gsc_impressions      BIGINT  YES None    None  None
              gsc_clicks      BIGINT  YES None    None  None
        gsc_sum_position      BIGINT  YES None    None  None
        gsc_avg_position      DOUBLE  YES None    None  None
           ga4_pageviews      BIGINT  YES None    None  None
            ga4_sessions      BIGINT  YES None    None  None
               ga4_users      BIGINT  YES None    None  None
    ga4_engaged_sessions      BIGINT  YES None    None  None
ga4_total_engagement_sec

**Unit of analysis:** One row represents one content item for one client on one reporting date (`report_date × client × content`).

**Time window:** I will use March 1–31, 2026, as the analysis slice. I will check the actual minimum and maximum dates and the available history for each client before interpreting results.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features
The following five candidate features are measured on the decision date:

1. `gsc_impressions` — known from search data recorded for that date.
2. `gsc_clicks` — known from search data recorded for that date.
3. `gsc_avg_position` — recorded average search position for that date; interpret only when GSC data is available.
4. `ga4_sessions` — recorded sessions for that date; use only when `ga4_data_available IS TRUE`.
5. `sessions_organic` — recorded organic sessions for that date; verify its availability and missingness before use.

These are candidate features because their measurements belong to the decision date, rather than a future outcome window.

### Label
`future_7d_click_decline` — a label to be constructed from future search clicks. It will indicate whether clicks in the next seven days decline compared with the preceding seven days. This label is not an existing column and must be calculated using a separate, explicitly defined query.

### Context
- `report_date` — defines the observation date.
- `client_hash_id` — identifies the pseudonymized client.
- `content_hash_id` — identifies the pseudonymized content item.
- `client_has_gsc`, `client_has_ga4`, `gsc_data_available`, `ga4_data_available` — indicate data availability and help interpret missing or zero-filled measurements.

### Excluded fields
- Future-window clicks and any columns derived from the label — excluded from features to prevent target leakage.
- `gsc_sum_position` — excluded from the initial feature set because `gsc_avg_position` is already selected; avoid redundant position measures.
- Individual AI referral columns such as `ai_chatgpt`, `ai_perplexity`, and `ai_gemini` — excluded from the initial five-feature set.
- Client and content IDs — used for grouping and context, not as predictive numeric features.

**Important:** The label definition is a proposed task definition, not a verified existing warehouse field. It must be implemented and validated before model training.

In [13]:

feature_check = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_impressions IS NULL) AS missing_impressions,
    COUNT(*) FILTER (WHERE gsc_clicks IS NULL) AS missing_clicks,
    COUNT(*) FILTER (WHERE gsc_avg_position IS NULL) AS missing_avg_position,
    COUNT(*) FILTER (WHERE ga4_sessions IS NULL) AS missing_ga4_sessions,
    COUNT(*) FILTER (WHERE sessions_organic IS NULL) AS missing_organic_sessions,
    COUNT(*) FILTER (
        WHERE ga4_data_available IS TRUE
    ) AS ga4_available_rows
FROM read_parquet('{march_url}')
"""

print(con.execute(feature_check).df().to_string(index=False))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 total_rows  missing_impressions  missing_clicks  missing_avg_position  missing_ga4_sessions  missing_organic_sessions  ga4_available_rows
    9841378                    0               0               6230317               3018741                   3018741              413966


In [14]:

flag_query = f"""
SELECT
    client_has_ga4,
    ga4_data_available,
    COUNT(*) AS rows
FROM read_parquet('{march_url}')
GROUP BY client_has_ga4, ga4_data_available
ORDER BY client_has_ga4, ga4_data_available
"""

print(con.execute(flag_query).df().to_string(index=False))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 client_has_ga4  ga4_data_available    rows
          False                <NA> 3018741
           True               False 6408671
           True                True  413966


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Verification results — March 2026

1. **Grain:** Grouped by `report_date`, `client_hash_id`, and `content_hash_id`. No duplicate combinations were found in the March partition.
2. **Time window and row count:** The partition contains 9,841,378 rows, from March 1, 2026, through March 31, 2026.
3. **Missing values:** No missing dates, client IDs, or content IDs were found. `gsc_avg_position` has 6,230,317 missing values, while `ga4_sessions` and `sessions_organic` each have 3,018,741 missing values.
4. **Availability flags:** 9,841,378 rows have `client_has_gsc = TRUE`. Only 413,966 rows have `ga4_data_available = TRUE`. `client_has_ga4` and `ga4_data_available` are not interchangeable.
5. **Interpretation:** GA4 features require explicit availability filtering. Missing search positions must not be automatically treated as zero or as poor rankings.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data limitations

- **Unequal coverage:** Clients may have different historical data availability, so March 2026 results may not generalize to every client or period.
- **Missing search position:** Many rows have no `gsc_avg_position`; the reason for each missing value has not yet been established.
- **Partial GA4 availability:** Only 413,966 rows in the tested partition are explicitly marked as GA4-available. Analyses using GA4 metrics must filter by the relevant availability flag.
- **Observational data:** Search and analytics metrics show associations, not proof that a content change caused a performance change.
- **Future-label limitation:** A future-click-decline label still needs to be constructed and validated using a separate future window. This notebook's March partition checks do not yet validate predictive performance.
- **Scope:** These checks cover March 2026 only. They do not establish data quality across every month or client.

**Permitted use:** The data can support descriptive analysis and carefully validated, directional decision support. It cannot, by itself, establish causation or guarantee future performance.

In [15]:

print(con.execute(f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date,
    COUNT(DISTINCT client_hash_id) AS distinct_clients,
    COUNT(DISTINCT content_hash_id) AS distinct_content_items
FROM read_parquet('{march_url}')
""").df().to_string(index=False))


 total_rows first_date  last_date  distinct_clients  distinct_content_items
    9841378 2026-03-01 2026-03-31                55                  331437


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.